<a href="https://colab.research.google.com/github/sayja-yug/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sayja-yug/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 1. Two Paper Findings + My Methodology Questions

## Purpose

This section practices the same kind of methodological thinking that I will
apply to my own Week-5 model.

I am not trying to grade the research paper. Instead, I am identifying two
specific findings and asking what evidence I would need to understand whether
the methodology supports those findings.

My lane is:

**Lane 2 — Refresh / Content Opportunity Scoring**

My Week-5 task was to identify content that may deserve review by predicting
whether observed April clicks would decline relative to March.

The important principle for this audit is:

> A strong metric is only meaningful when the label definition and validation
> design support the claim being made.

---

## Finding 1 — Model performance / predictive result

### Finding

The paper reports a model result and uses evaluation metrics to demonstrate
performance.

### My methodology question

**Where exactly does the label come from, and is the label independently
observed rather than being derived from information that also enters the model?**

I would want to verify:

- how the positive and negative examples were defined;
- whether the label was available only after the prediction window;
- whether any label-derived field was included as a feature;
- whether the label definition matches the real decision being supported;
- whether the evaluation data was kept separate from model development.

### Why this matters for my own model

In my Week-5 model, I defined:

`future_decline = 1`

when April clicks were at least 20% lower than March clicks.

Therefore, April information must only be used to create the observed target.
April clicks, April impressions, April position, or the final decline calculation
must not become model features.

This is an important distinction between:

**predicting a future outcome**

and

**describing an outcome that has already happened.**

My Week-6 audit will explicitly check this boundary.

---

## Finding 2 — Validation / generalization result

### Finding

The paper reports evaluation results intended to show how well the proposed
method performs beyond the data used to develop it.

### My methodology question

**Does the validation design actually test the generalization claim being made?**

I would want to know:

- whether train and validation examples can belong to the same underlying
  client/entity;
- whether related observations can appear on both sides of the split;
- whether the split is grouped or time-aware when the data structure requires it;
- whether preprocessing was fitted only on the training data;
- whether the reported metric is calculated on genuinely held-out data.

### Why this matters for my own model

My FlyRank data contains repeated observations for the same client and content.

A random row-level split could therefore make the task easier than the real
deployment situation because the same client's patterns could appear in both
training and testing.

In Week-5 I already used a client-grouped split.

In Week-6 I will explicitly compare the validation design and verify that the
evaluation remains honest.

---

## What I am taking from the paper

The main lesson I am applying is not simply "use a better model."

The lesson is:

1. Define the label independently.
2. Make the validation design match the claim.
3. Check for leakage before trusting the metric.
4. Inspect real failures instead of relying only on a single score.
5. Rewrite claims when the evidence is weaker than the original wording.

For my Lane-2 project, the model is decision-support for prioritizing content
review. It does not prove that refreshing a page will cause recovery.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 2. My Model Under an Honest Split — Before / After

## Objective

The purpose of this section is to test whether my Week-5 model performance
remains supported when the validation design reflects the structure of the
data.

My Lane-2 decision is:

> Which content pages should be reviewed first because they show evidence of
> future click decline?

The prediction setup is:

**March 2026 features → April 2026 observed outcome**

---

## Feature window

March 2026 is the decision/feature window.

The model can use information known by the end of March:

- March impressions
- March clicks
- March average position

The modelling grain is:

**one row per client × content item**

---

## Future outcome

April 2026 is the future outcome window.

The target is:

`future_decline = 1`

when April clicks are at least 20% lower than March clicks.

April information is used only to create the observed target.

It must never be used as a model feature.

---

## Baseline

The Week-4 `baseline_score` is used only as the comparison system.

It is NOT a feature of the machine-learning model.

This prevents the model from simply learning the previous decision rule.

---

## Before validation design

The Week-5 model used a client-grouped train/test split.

This means complete clients are assigned to either training or testing,
rather than allowing the same client to appear in both.

This is preferable to a simple random row split because the data contains
many observations belonging to the same client.

---

## After / audit design

For this validation audit, I will verify the grouped split explicitly.

The key condition is:

**No client may appear in both training and testing.**

If the same client appears in both groups, the validation result may be
optimistic because the model can benefit from client-specific patterns that
it has already seen.

---

## What I will compare

I will compare:

1. Week-4 baseline
2. Week-5 Logistic Regression
3. Week-5 Random Forest

using the same held-out test rows and the same target.

The primary ranking metrics are:

- Precision@100
- Precision@500
- Precision@1000
- Average Precision

These metrics are more relevant than accuracy because the actual use case is
a ranked review queue rather than classification of every page equally.

---

## What makes the validation honest?

An honest validation requires:

- no future April features;
- no `future_decline` feature;
- no `baseline_score` feature;
- no `reason_code` feature;
- no `action_label` feature;
- no client overlap between training and testing;
- evaluation on held-out data;
- comparison against the Week-4 baseline on the same test population.

The goal is not to make the model look better.

The goal is to determine whether the observed improvement survives a
validation design that matches the real decision problem.

In [9]:
# ============================================================
# ENVIRONMENT CHECK
# ============================================================

import pandas as pd
import pyarrow as pa
import datasets
import huggingface_hub

print("=" * 70)
print("ENVIRONMENT CHECK")
print("=" * 70)

print("Pandas          :", pd.__version__)
print("PyArrow         :", pa.__version__)
print("Datasets        :", datasets.__version__)
print("HuggingFace Hub :", huggingface_hub.__version__)

print("\nTesting PyArrow...")

test_df = pd.DataFrame({
    "x": [1, 2, 3],
    "y": ["a", "b", "c"]
})

test_df.to_parquet(
    "/content/test_arrow.parquet",
    index=False
)

test_back = pd.read_parquet(
    "/content/test_arrow.parquet"
)

display(test_back)

print("\n✅ Pandas + PyArrow working correctly.")
print("✅ Datasets imported correctly.")
print("✅ Environment is ready.")

AttributeError: module 'pyarrow.lib' has no attribute 'Decimal32Type'

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# WEEK 6 — SECTION 2
# HONEST VALIDATION DATA SETUP
#
# Lane 2: Refresh / Content Opportunity Scoring
#
# March 2026 = feature / decision window
# April 2026 = future observed outcome window
# ============================================================

import os
import numpy as np
import pandas as pd

from google.colab import userdata
from datasets import load_dataset

print("=" * 75)
print("WEEK 6 — SECTION 2: HONEST VALIDATION")
print("=" * 75)


# ============================================================
# 1. LOAD HUGGING FACE TOKEN
# ============================================================

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets. "
        "Add your Hugging Face READ token as HF_TOKEN."
    )

print("✅ Hugging Face token loaded.")


# ============================================================
# 2. LOAD WEEK-4 BASELINE
# ============================================================

baseline_paths = [
    "/content/baseline_action_score.csv",
    "/content/work/outputs/baseline_action_score.csv"
]

baseline_path = None

for path in baseline_paths:
    if os.path.exists(path):
        baseline_path = path
        break

if baseline_path is None:
    raise FileNotFoundError(
        "baseline_action_score.csv was not found in Colab."
    )

baseline_df = pd.read_csv(baseline_path)

print("\n" + "=" * 75)
print("WEEK-4 BASELINE")
print("=" * 75)

print("Baseline path:", baseline_path)
print("Baseline rows:", len(baseline_df))

required_baseline_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "baseline_score",
    "reason_code",
    "action_label"
]

missing_baseline = [
    c for c in required_baseline_columns
    if c not in baseline_df.columns
]

if missing_baseline:
    raise ValueError(
        "Missing Week-4 baseline columns: "
        + str(missing_baseline)
    )

baseline_df["report_date"] = pd.to_datetime(
    baseline_df["report_date"],
    errors="coerce"
)

baseline_df = baseline_df[
    baseline_df["report_date"].between(
        "2026-03-01",
        "2026-03-31"
    )
].copy()

print("March baseline rows:", len(baseline_df))
print("✅ Week-4 baseline loaded.")


# ============================================================
# 3. LOAD FLYRANK DAILY WAREHOUSE
# ============================================================

print("\n" + "=" * 75)
print("LOADING FLYRANK DAILY WAREHOUSE")
print("=" * 75)

warehouse_dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    token=HF_TOKEN
)

print("✅ Warehouse dataset loaded.")
print("Total warehouse rows:", len(warehouse_dataset))


# ============================================================
# 4. SELECT ONLY REQUIRED COLUMNS
# ============================================================

required_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

missing_columns = [
    c for c in required_columns
    if c not in warehouse_dataset.column_names
]

if missing_columns:
    raise ValueError(
        "Missing warehouse columns: "
        + str(missing_columns)
    )

warehouse_df = warehouse_dataset.select_columns(
    required_columns
).to_pandas()

print("✅ Required warehouse columns selected.")
print("Warehouse dataframe shape:", warehouse_df.shape)


# ============================================================
# 5. STANDARDIZE DATE
# ============================================================

warehouse_df["report_date"] = pd.to_datetime(
    warehouse_df["report_date"],
    errors="coerce"
)

warehouse_df = warehouse_df.dropna(
    subset=[
        "report_date",
        "client_hash_id",
        "content_hash_id"
    ]
).copy()


# ============================================================
# 6. CREATE MARCH FEATURE WINDOW
# ============================================================

march_df = warehouse_df[
    warehouse_df["report_date"].between(
        "2026-03-01",
        "2026-03-31"
    )
].copy()

print("\nMarch rows:", len(march_df))

if march_df.empty:
    raise ValueError("March 2026 data is empty.")


# ============================================================
# 7. CREATE APRIL OUTCOME WINDOW
# ============================================================

april_df = warehouse_df[
    warehouse_df["report_date"].between(
        "2026-04-01",
        "2026-04-30"
    )
].copy()

print("April rows:", len(april_df))

if april_df.empty:
    raise ValueError("April 2026 data is empty.")


# ============================================================
# 8. DEFINE MODEL GRAIN
# ============================================================

grain = [
    "client_hash_id",
    "content_hash_id"
]

print("\nFeature grain:")
print("one row = client × content")


# ============================================================
# 9. AGGREGATE MARCH FEATURES
# ============================================================

march_features_df = (
    march_df
    .groupby(grain, as_index=False)
    .agg(
        march_impressions=(
            "gsc_impressions",
            "sum"
        ),
        march_clicks=(
            "gsc_clicks",
            "sum"
        ),
        march_avg_position=(
            "gsc_avg_position",
            "mean"
        )
    )
)

print("\n" + "=" * 75)
print("MARCH FEATURES")
print("=" * 75)

print("Rows:", len(march_features_df))

display(
    march_features_df.head()
)


# ============================================================
# 10. AGGREGATE APRIL OUTCOME
# ============================================================

april_outcome_df = (
    april_df
    .groupby(grain, as_index=False)
    .agg(
        april_impressions=(
            "gsc_impressions",
            "sum"
        ),
        april_clicks=(
            "gsc_clicks",
            "sum"
        ),
        april_avg_position=(
            "gsc_avg_position",
            "mean"
        )
    )
)

print("\n" + "=" * 75)
print("APRIL OUTCOME")
print("=" * 75)

print("Rows:", len(april_outcome_df))

display(
    april_outcome_df.head()
)


# ============================================================
# 11. JOIN MARCH FEATURES TO APRIL OUTCOME
# ============================================================

comparison_base_df = march_features_df.merge(
    april_outcome_df,
    on=grain,
    how="inner"
)

print("\nJoined March + April rows:",
      len(comparison_base_df))


# ============================================================
# 12. CREATE OBSERVED FUTURE TARGET
# ============================================================

comparison_base_df["future_decline"] = (
    comparison_base_df["april_clicks"]
    <=
    comparison_base_df["march_clicks"] * 0.80
).astype(int)


# ============================================================
# 13. APPLY MARCH ACTIVITY FILTER
# ============================================================

eligible_df = comparison_base_df[
    (comparison_base_df["march_impressions"] >= 100)
    &
    (comparison_base_df["march_clicks"] >= 5)
].copy()

print("\nEligible rows:", len(eligible_df))


# ============================================================
# 14. ATTACH WEEK-4 BASELINE
# ============================================================

baseline_for_join = baseline_df[
    [
        "client_hash_id",
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].copy()


# Keep strongest baseline score per client-content pair
baseline_for_join = (
    baseline_for_join
    .sort_values(
        "baseline_score",
        ascending=False
    )
    .drop_duplicates(
        subset=grain,
        keep="first"
    )
)


comparison_df = eligible_df.merge(
    baseline_for_join,
    on=grain,
    how="inner"
)


# ============================================================
# 15. FINAL MODEL FEATURE DEFINITION
# ============================================================

MODEL_FEATURES = [
    "march_impressions",
    "march_clicks",
    "march_avg_position"
]

TARGET = "future_decline"

GROUP_COLUMN = "client_hash_id"


# ============================================================
# 16. REMOVE INVALID FEATURE VALUES
# ============================================================

comparison_df = comparison_df.replace(
    [np.inf, -np.inf],
    np.nan
)

comparison_df = comparison_df.dropna(
    subset=MODEL_FEATURES + [TARGET]
).copy()


# ============================================================
# 17. LEAKAGE CHECK
# ============================================================

forbidden_features = [
    "baseline_score",
    "reason_code",
    "action_label",
    "april_impressions",
    "april_clicks",
    "april_avg_position",
    "future_decline"
]

leakage_found = [
    feature
    for feature in MODEL_FEATURES
    if feature in forbidden_features
]

if leakage_found:
    raise RuntimeError(
        "LEAKAGE FOUND: " + str(leakage_found)
    )

print("\n✅ No future or baseline decision outputs are model features.")


# ============================================================
# 18. TARGET CHECK
# ============================================================

print("\n" + "=" * 75)
print("TARGET DISTRIBUTION")
print("=" * 75)

display(
    comparison_df[TARGET]
    .value_counts()
    .sort_index()
    .rename_axis(TARGET)
    .to_frame("n")
)


# ============================================================
# 19. FINAL OBJECT CHECK
# ============================================================

print("\n" + "=" * 75)
print("SECTION 2 OBJECT CHECK")
print("=" * 75)

print("comparison_df rows:",
      len(comparison_df))

print("Model features:",
      MODEL_FEATURES)

print("Target:",
      TARGET)

print("Grouping column:",
      GROUP_COLUMN)

print("\nRequired columns:")

required_final_columns = [
    "client_hash_id",
    "content_hash_id",
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "future_decline",
    "baseline_score"
]

for col in required_final_columns:
    if col in comparison_df.columns:
        print("✅", col)
    else:
        print("❌", col)


# ============================================================
# 20. SHOW FINAL DATA
# ============================================================

print("\nFinal comparison dataset:")

display(
    comparison_df[
        required_final_columns
    ].head(10)
)


# ============================================================
# 21. FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("SECTION 2 DATA SETUP COMPLETE")
print("=" * 75)

print("Decision window : March 2026")
print("Outcome window  : April 2026")
print("Feature grain   : client × content")
print("Target          : future_decline")
print("Features        :", MODEL_FEATURES)
print("Baseline        : Week-4 baseline_score")
print("Baseline feature: NO")
print("April features  : NO")

print("\nCreated dataframes:")
print("✅ baseline_df")
print("✅ warehouse_df")
print("✅ march_df")
print("✅ april_df")
print("✅ march_features_df")
print("✅ april_outcome_df")
print("✅ comparison_base_df")
print("✅ eligible_df")
print("✅ comparison_df")

print("\n✅ Ready for the remaining Section 2 validation code.")

AttributeError: module 'pyarrow.lib' has no attribute 'Decimal32Type'

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.